# Verification register

Run all cells to see what is calculated, what fails, and what is still missing.
The register preserves **373 requirements and 677 activities** from the
requirements sheet, including unresolved values marked TBD or TBR.

A calculation passing its limit does **not** close a requirement. Closure
needs reviewed passing evidence for every activity and resolved TBD/TBR values.


In [1]:
import pandas as pd
from IPython.display import display, Markdown

from hyperion.workspace import current_flight
from hyperion.verification import ROOT, evaluate, export, requirement_status

pd.set_option("display.max_colwidth", 100)


## 1. Run the current checks

| Result | Meaning | Next step |
|---|---|---|
| `SCREEN_PASS` | Calculation meets the supplied limit | Validate assumptions and review evidence |
| `SCREEN_FAIL` | Calculation misses the supplied limit | Review the design, inputs, and limit |
| `CALCULATED` | Result exists without an acceptance rule | Supply an approved limit and source |
| `MISSING_INPUTS` | Calculation needs more data | Supply the named inputs |
| `EVIDENCE_REQUIRED` | Test, inspection, or analysis record is missing | Add the reviewed evidence |
| `INVALID_INPUTS` / `INVALID_EVIDENCE` | Data or record cannot be used | Fix the reported issue |
| `STALE_EVIDENCE` | Record refers to older inputs or wording | Review it against the current version |


In [2]:
results, details = evaluate(context=current_flight())
requirements = requirement_status(results)

activity_counts = (
    results.status.value_counts().rename_axis("Activity result").to_frame("Count")
)
requirement_counts = (
    requirements.status.value_counts()
    .rename_axis("Requirement result")
    .to_frame("Count")
)
display(activity_counts)
display(requirement_counts)


,Count
Activity result,
EVIDENCE_REQUIRED,431
MISSING_INPUTS,227
CALCULATED,7
SCREEN_FAIL,6
SCREEN_PASS,6


,Count
Requirement result,
OPEN,367
FAILED_OR_INVALID,6


## 2. Start with failures

The table below shows failed checks and unusable records. Passing calculations
still require review. The full export also retains stale and missing evidence.


In [3]:
failure_statuses = [
    "SCREEN_FAIL",
    "REVIEWED_FAIL",
    "INVALID_INPUTS",
    "INVALID_EVIDENCE",
]
failures = results[results.status.isin(failure_statuses)]
columns = {
    "activity_id": "Activity ID",
    "activity": "Required work",
    "status": "Result",
    "detail": "Basis or issue",
}
display(failures[list(columns)].rename(columns=columns))


,Activity ID,Required work,Result,Basis or issue
0,MIS.0.01.A01,Integrated trajectory simulation from launch to TMI.,SCREEN_FAIL,600s coast and extra1.5s-equivalent restart debit are screening assumptions. Shared 6DOF prograd...
22,SYS.1.02.A01,"Integrated ascent, coast and finite-burn TMI simulation.",SCREEN_FAIL,600s coast and extra1.5s-equivalent restart debit are screening assumptions. Shared 6DOF prograd...
37,SYS.1.09.A02,Integrated finite-burn trajectory simulation.,SCREEN_FAIL,600s coast and extra1.5s-equivalent restart debit are screening assumptions. Shared 6DOF prograd...
85,PERF.2.13.A02,Control and aerodynamic moment balance.,SCREEN_FAIL,Requirement numeric screen only; model-validation and open source conflicts remain.
291,PROP.3.SYS.04.A02,First-stage engine thrust summation.,SCREEN_FAIL,3 source Raptor3 engines at2255kN each; source sea-level basis remains to be substantiated.
390,PROP.3.TANK.02.A01,Ullage and loaded-liquid volume calculations.,SCREEN_FAIL,"Current Mlox-Hlox Vehicle Sizing G4:H5,F19:G20,B31:C32; Vehicle Mass E29/E44 totals plus fairing..."


## 3. Inspect one requirement

Change `REQUIREMENT` below. Read the exact requirement, source location, and
required activities before changing its inputs or attaching evidence.


In [4]:
REQUIREMENT = "PERF.2.13"
selected = results[results.requirement_id.eq(REQUIREMENT)]

if selected.empty:
    print("No matching requirement. Check data/activities.csv for available IDs.")
else:
    requirement = selected.iloc[0]
    display(Markdown(f"**{REQUIREMENT}:** {requirement['statement']}"))
    print("Source:", requirement["source"])
    print("Unresolved values:", requirement["open_parameters"] or "None listed")
    columns = {
        "activity_id": "Activity ID",
        "activity": "Required work",
        "kernel": "Calculation",
        "status": "Result",
        "detail": "Basis or missing inputs",
    }
    display(selected[list(columns)].rename(columns=columns).fillna("External evidence"))


**PERF.2.13:** The LV shall maintain a controllability ratio of at least 2.

Source: 2 LV Performance!H14
Unresolved values: None listed


,Activity ID,Required work,Calculation,Result,Basis or missing inputs
84,PERF.2.13.A01,Actuator limit and mass-property inspection.,External evidence,EVIDENCE_REQUIRED,"Add the CAD/FEA, specialist analysis, inspection, demonstration or test record for this activity..."
85,PERF.2.13.A02,Control and aerodynamic moment balance.,flight_summary,SCREEN_FAIL,Requirement numeric screen only; model-validation and open source conflicts remain.


## 4. Add inputs or evidence, then rerun

1. Find the activity ID and its calculation in [activities.csv](data/activities.csv).
2. Look up required arguments in [calculation_inputs.csv](data/calculation_inputs.csv).
3. Add sourced values and acceptance rules to
   [verification_inputs.json](config/verification_inputs.json). An activity ID
   entry overrides shared inputs for that calculation.
4. Add reviewed tests, inspections, or specialist analyses using the
   [evidence guide](evidence/README.md).
5. Rerun this notebook. The cell below saves the complete tables to `results/current/`.

After a requirements revision, confirm OneDrive is synced and follow the
[import steps in the README](README.md#update-the-requirements). The importer
reads the workbook; it does not edit it or synchronize OneDrive.


In [5]:
output_folder = export(results, details, ROOT / "results/current")
print(
    f"Saved {len(results)} activities and {len(requirements)} requirements to results/current/."
)
print(
    "Open activity_results.csv for the full work list; requirement_status.csv for closure status."
)


Saved 677 activities and 373 requirements to results/current/.
Open activity_results.csv for the full work list; requirement_status.csv for closure status.
